# Reproduce unsupervised Figures 9-11: Tennessee Eastman PCA example

This notebook reproduces the Tennessee Eastman Process (TEP) PCA example from the paper using the attached Excel dataset. It follows the JMP `measurement vars` column group plus the added `Random Normal` SNF, then recomputes PCA in Python with `pyphi-mvda`.

Figure 9(a) in the paper is external P&ID artwork adapted from another source. Per reproduction scope, this notebook recreates the data-driven process-trend panel and documents the omitted external artwork.

## Reference and library

**Main article:** Vallerio, M., del Rio Chanona, A., & Navarro-Brull, F. J. (2026). *All you need is noise - from feature selection to explainable industrial AI*. **Digital Chemical Engineering, 18**, 100290. https://doi.org/10.1016/j.dche.2026.100290

**PCA library used:** this notebook fits PCA models with `pyphi-mvda`, the Python MVDA library from Salvador Garcia Munoz: https://github.com/salvadorgarciamunoz/pyphi. The package is imported as `from pyphi import calc as phi`, and PCA is fitted with `phi.pca(...)` using autoscaling (`mcs=True`) to match JMP's PCA-on-correlations workflow.

**Paper idea reproduced here:** the article adds a Synthetic Noise Feature (SNF) to the variable set before fitting PCA. Principal components are retained only until the first component where the SNF becomes the dominant loading. Components at and after that point are treated as carrying no more structured information than injected noise.

In [ ]:
# Self-contained runtime setup for reproducing the paper figures.
import importlib.util
import subprocess
import sys


def ensure(import_name, package_name=None):
    package_name = package_name or import_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


for import_name, package_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("openpyxl", "openpyxl"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("scipy", "scipy"),
    ("pyphi", "pyphi-mvda"),
]:
    ensure(import_name, package_name)

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyphi import calc as phi

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 10,
})
sns.set_theme(style="whitegrid", context="notebook", font_scale=0.92)


def find_unsupervised_root():
    """Locate the 02_unsupervised folder from this notebook or current working directory."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    fallback = Path("/Users/b42549592/Documents/GitHub/all-you-need-is-noise/02_unsupervised")
    candidates.extend([fallback, *fallback.parents])
    for candidate in candidates:
        if candidate.name == "02_unsupervised" and (candidate / "01_density_dataset").exists() and (candidate / "02_TEP").exists():
            return candidate
        nested = candidate / "02_unsupervised"
        if (nested / "01_density_dataset").exists() and (nested / "02_TEP").exists():
            return nested
    raise FileNotFoundError("Could not locate the 02_unsupervised folder")


UNSUPERVISED_ROOT = find_unsupervised_root()

PALETTE = {
    "real_1": "#1f77b4",
    "real_2": "#111111",
    "noise": "#d62728",
    "pc1": "#1f77b4",
    "pc2": "#d62728",
    "pc3": "#2ca02c",
    "pc4": "#9467bd",
    "pc5": "#ff7f0e",
}


def pca_with_pyphi(data, obs_col, feature_cols, n_components):
    """Fit pyphi PCA and return scores, loadings, and explained variance tables."""
    X = pd.concat([
        data[obs_col].astype(str).rename("obs"),
        data[feature_cols].apply(pd.to_numeric, errors="coerce"),
    ], axis=1)
    model = phi.pca(X, n_components, mcs=True, force_nipals=True, shush=True)
    pc_names = [f"PC{i}" for i in range(1, n_components + 1)]
    scores = pd.DataFrame(model["T"], columns=pc_names, index=data.index)
    loadings = pd.DataFrame(model["P"], index=model["varidX"], columns=pc_names)
    explained = pd.DataFrame({
        "PC": pc_names,
        "explained_percent": np.asarray(model["r2x"])[:n_components] * 100,
    })
    explained["cumulative_percent"] = explained["explained_percent"].cumsum()
    return model, scores, loadings, explained

In [ ]:
DATA_PATH = UNSUPERVISED_ROOT / "02_TEP" / "Tennesse MDMSPC - PCA - Eastman Chemical.xlsx"
raw = pd.read_excel(DATA_PATH)

# JMP's `measurement vars` column group corresponds to these 22 process measurements.
measurement_cols = list(raw.columns[4:26])
feature_cols = measurement_cols + ["Random Normal"]
model, scores, loadings, explained = pca_with_pyphi(raw, "Row number", feature_cols, 12)
# JMP plots the PCA loading matrix as eigenvectors scaled by sqrt(eigenvalue).
eigenvalues = explained["explained_percent"].to_numpy() / 100 * len(feature_cols)
loading_matrix = loadings.multiply(np.sqrt(eigenvalues), axis=1)
abs_loadings = loading_matrix.abs()
first_noise_dominant_pc = next(
    pc for pc in abs_loadings.columns
    if abs_loadings[pc].idxmax() == "Random Normal"
)
cutoff_pc_number = int(first_noise_dominant_pc.replace("PC", ""))

print(f"Loaded {len(raw)} TEP observations and {len(feature_cols)} PCA variables")
print(f"First SNF-dominant component: {first_noise_dominant_pc}")
display(raw[["Row number", "type"] + feature_cols[:5]].head())
display(explained.head(12).round(3))

## Validation of the SNF cutoff

The notebook uses the same process-measurement subset described in the JMP file as `measurement vars`, plus `Random Normal`. With this selection, the synthetic noise feature first dominates PC5, matching the paper.

In [ ]:
# Identify the strongest loading in each early PC and verify the paper's SNF cutoff.
top_by_pc = pd.DataFrame({
    "PC": abs_loadings.columns[:8],
    "top_variable": [abs_loadings[pc].idxmax() for pc in abs_loadings.columns[:8]],
    "top_abs_loading": [abs_loadings[pc].max() for pc in abs_loadings.columns[:8]],
    "random_normal_abs_loading": [abs_loadings.loc["Random Normal", pc] for pc in abs_loadings.columns[:8]],
})
display(top_by_pc.round(4))
assert first_noise_dominant_pc == "PC5"

## Figure 9: TEP process trends

**Article explanation:** Figure 9 introduces the industrial TEP use case. The dataset contains a plant response to a load change in the A-feed stream; many process variables respond dynamically before settling toward a new operating point. This provides a realistic multivariate setting for applying the SNF/PCA rule.

In [ ]:
# Order variables to follow the paper/JMP trend-panel emphasis.
trend_order = [
    "A_Feed_S1", "Total_Feed_S4", "Stripper_Temp", "Stripper_Steam_Flow", "Compressor_Work",
    "Stripper_Pressure", "Reactor_Pressure", "Prod_Sep_Pressure", "Product_Sep_Temp", "Purge_Rate_S9",
    "Reactor_Cooling_Water_Outlet_Temp", "Reactor_Level", "E_Feed_S3",
    "Separator_Cooling_Water_Outlet_Temp", "Reactor_Feed_Rate_S6", "Product_Sep_Level",
    "Prod_Sep_Underflow_S10", "D_Feed_S2", "Stripper_Underflow_S11", "Random Normal",
    "Stripper_Level", "Recycle_Flow_S8", "Reactor_Temperature",
]
trend_order = [col for col in trend_order if col in raw.columns]
trend = raw[["Row number"] + trend_order].copy()

# Min-max scale each variable so all process trajectories fit into stacked lanes.
scaled = trend[trend_order].apply(pd.to_numeric, errors="coerce")
scaled = (scaled - scaled.min()) / (scaled.max() - scaled.min())

fig, ax = plt.subplots(figsize=(13.5, 9.4), constrained_layout=True)
lane_gap = 1.0
for i, col in enumerate(trend_order):
    color = PALETTE["noise"] if col == "Random Normal" else plt.cm.tab20(i % 20)
    y = scaled[col] + i * lane_gap
    ax.plot(trend["Row number"], y, color=color, lw=0.9, alpha=0.95)

# Mark the transition from historic to current observations when available.
transition = raw.loc[raw["type"].astype(str).eq("current"), "Row number"].min()
if pd.notna(transition):
    ax.axvline(transition, color="#444444", lw=1.1, linestyle="--", alpha=0.75)
    ax.text(transition, len(trend_order) * lane_gap, "current period", rotation=90,
            va="top", ha="right", color="#444444", fontsize=9)

ax.set_yticks(np.arange(len(trend_order)) * lane_gap + 0.5)
ax.set_yticklabels(trend_order, fontsize=8)
ax.set_xlabel("Row number")
ax.set_ylabel("")
ax.set_title("Figure 9 reproduction: TEP process-variable trends", loc="left", fontweight="bold")
ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", alpha=0.08)
plt.show()

**Figure 9 caption.** Visualization of the TEP use case. This reproduction shows trends of selected variables from the attached TEP dataset. The paper also includes a P&ID panel adapted from Fortela and Mikolajczyk (2023), which is intentionally omitted here because it is external artwork.

## Figure 10: explained variance and loading heatmap

**Article explanation:** Figure 10 applies the SNF cutoff rule to the TEP PCA model. PC5 is the first component where the synthetic noise feature becomes the dominant variable. Therefore PC5 and subsequent components are treated as non-relevant for downstream analysis, while PC1-PC4 are retained.

In [ ]:
# Show the first eight PCs in the heatmap and the first twelve in the variance plot.
pcs_to_show = [f"PC{i}" for i in range(1, 9)]
fig = plt.figure(figsize=(14.2, 8.1), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.45])
ax_var = fig.add_subplot(gs[0, 0])
ax_heat = fig.add_subplot(gs[0, 1])

# Highlight PC5 onward because PC5 is the first SNF-dominated component.
expl = explained.iloc[:12].copy()
xpos = np.arange(len(expl))
colors = ["#4c78a8" if i < cutoff_pc_number else "#d95f5f" for i in range(1, len(expl) + 1)]
ax_var.bar(xpos, expl["explained_percent"], color=colors, label="Individual")
ax_var.plot(xpos, expl["cumulative_percent"], color="#222222", marker="o", lw=2.2,
            linestyle="--", label="Cumulative")
ax_var.axvspan(cutoff_pc_number - 1 - 0.5, len(expl) - 0.5, color="#d95f5f", alpha=0.10)
ax_var.axvline(cutoff_pc_number - 1 - 0.5, color=PALETTE["noise"], lw=1.6, linestyle=":")
ax_var.text(cutoff_pc_number - 1.35, 88, "SNF cutoff", color=PALETTE["noise"], rotation=90,
            va="top", ha="center", fontsize=9)
ax_var.set_xticks(xpos, expl["PC"], rotation=45)
ax_var.set_ylim(0, 100)
ax_var.set_ylabel("Explained variance [%]")
ax_var.set_title("(a) Individual and cumulative explained variance", loc="left", fontweight="bold")
ax_var.legend(loc="lower right", frameon=True)

# Order variables by total loading over the displayed PCs to make the heatmap readable.
heat_order = abs_loadings[pcs_to_show].assign(total=abs_loadings[pcs_to_show].sum(axis=1)) \
    .sort_values("total", ascending=True).drop(columns="total").index
sns.heatmap(abs_loadings.loc[heat_order, pcs_to_show], ax=ax_heat, cmap="magma", vmin=0,
            cbar_kws={"label": "Abs(loading)"}, linewidths=0.15, linecolor="white")
ax_heat.set_title("(b) Absolute loading heatmap", loc="left", fontweight="bold")
ax_heat.set_xlabel("Principal components")
ax_heat.set_ylabel("Process variables")
for tick in ax_heat.get_xticklabels():
    if int(tick.get_text().replace("PC", "")) >= cutoff_pc_number:
        tick.set_color(PALETTE["noise"])
        tick.set_fontweight("bold")

fig.suptitle("Figure 10 reproduction: PC5 is the first SNF-dominated component", fontweight="bold")
plt.show()

**Figure 10 caption.** PC5 emerges as the first principal component dominated by the synthetic-noise feature, establishing a cutoff for dimensionality reduction. Panel (a) shows cumulative and individual explained variance; panel (b) shows absolute loadings with variables on the y-axis and PCs on the x-axis.

## Figure 11: cumulative absolute loadings across PC1-PC5

**Article explanation:** Figure 11 converts the heatmap logic into a ranked variable-selection view. Variables are ordered by their cumulative absolute loading across PC1-PC5. The SNF acts as a cutoff: variables below or near that cumulative loading are lower-information variables for this PCA analysis.

In [ ]:
# Rank variables by cumulative absolute loading across retained PCs plus the cutoff PC.
stack_pcs = [f"PC{i}" for i in range(1, cutoff_pc_number + 1)]
stacked = abs_loadings[stack_pcs].copy()
stacked["total"] = stacked.sum(axis=1)
stacked = stacked.sort_values("total", ascending=True)
noise_total = stacked.loc["Random Normal", "total"]

fig, ax = plt.subplots(figsize=(11.4, 8.5), constrained_layout=True)
left = np.zeros(len(stacked))
pc_colors = [PALETTE["pc1"], "#2ca02c", PALETTE["pc2"], PALETTE["pc4"], PALETTE["pc5"]]
for pc, color in zip(stack_pcs, pc_colors):
    ax.barh(stacked.index, stacked[pc], left=left, color=color, label=pc, height=0.62)
    left += stacked[pc].to_numpy()

# The Random Normal cumulative loading is used as the variable-selection threshold.
ax.axvline(noise_total, color=PALETTE["noise"], lw=1.8, linestyle=":")
ax.text(noise_total + 0.015, len(stacked) - 2.0, "Random Normal threshold", color=PALETTE["noise"], fontsize=9)

for label in ax.get_yticklabels():
    if label.get_text() == "Random Normal":
        label.set_color(PALETTE["noise"])
        label.set_fontweight("bold")

ax.set_xlabel("Cumulative sum of Abs(PC loading), PC1-PC5")
ax.set_ylabel("")
ax.set_title("Figure 11 reproduction: variable ranking by cumulative PCA loading", loc="left", fontweight="bold")
ax.legend(loc="lower right", ncol=5, frameon=True, title="")
plt.show()

**Figure 11 caption.** Ranking variables by cumulative PCA loading places the synthetic-noise tag as a cutoff while spotlighting the key TEP drivers. Bars show cumulative absolute loadings across PC1-PC5, with colors indicating each PC contribution.